# imports

In [29]:
from preprocess import get_data
from model_utils import export_model_data_json, get_auc_roc
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split

In [30]:
new_df = get_data()
new_df

,VITESSE_MA,METEO_beau,METEO_chute de neige,METEO_couvert,METEO_grêle,METEO_pluie,METEO_pluie givrante,ETAT_ROUTE_humide,ETAT_ROUTE_mouillé,ETAT_ROUTE_route enneigée,ETAT_ROUTE_sèche,ETAT_ROUTE_verglacée,LUMIERE_inconnu,LUMIERE_jour,LUMIERE_nuit,HEURE_SIN,HEURE_COS,VICTIME
0,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.866,-0.500,False
1,80.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.000,-1.000,False
2,50.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.707,0.707,False
3,50.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.000,0.000,False
4,80.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-0.866,-0.500,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18586,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.259,-0.966,False
18587,120.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.707,0.707,False
18588,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.259,-0.966,False
18589,50.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.500,-0.866,False


In [31]:
X_train, X_temp, y_train, y_temp = train_test_split(
    new_df.drop(columns=["VICTIME"]),
    new_df[["VICTIME"]],
    test_size=0.2,
    random_state=42,
)

In [32]:
def find_best_hyperparameters(
    regressor, param_grid, X_train, y_train, scoring="accuracy", sample_weights=None
):
    grid_search = GridSearchCV(
        regressor,
        param_grid,
        cv=5,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1,
        verbose=0,
    )  # use all the cores of the CPU
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)
    print("Best hyperparameters: {}".format(grid_search.best_params_))
    print("Best score ({}): {:.2f}".format(scoring, grid_search.best_score_))
    return grid_search.best_estimator_

In [33]:
def print_cross_validation_results(regressor, X_train, y_train):
    scoring = ["accuracy", "f1", "roc_auc"]    
    cv_results = cross_validate(
        regressor,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        n_jobs=-1,
        verbose=0,
        return_train_score=True,
    )
    print("Train Accuracy: {:.2f}".format(cv_results["train_accuracy"].mean()))
    print("Validation Accuracy: {:.2f}".format(cv_results["test_accuracy"].mean()))
    print("Train F1-Score: {:.2f}".format(cv_results["train_f1"].mean()))
    print("Validation F1-Score: {:.2f}".format(cv_results["test_f1"].mean()))
    print("Train AUC-ROC: {:.2f}".format(cv_results["train_roc_auc"].mean()))
    print("Validation AUC-ROC: {:.2f}".format(cv_results["test_roc_auc"].mean()))

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    new_df.drop(columns=["VICTIME"]),
    new_df[["VICTIME"]],
    test_size=0.4,
    random_state=42,
)


param_grid = {
    "n_estimators": [10, 100, 1000],
    "learning_rate": [0.1, 0.05, 0.01, 0.001],
}

gb_clf = GradientBoostingClassifier(
    n_estimators=100, learning_rate=1.0, max_depth=1, random_state=0
)

gb_clf = find_best_hyperparameters(
    gb_clf,
    param_grid,
    X_train,
    y_train,
    scoring="roc_auc"
)
print_cross_validation_results(gb_clf, X_train, y_train)

c:\Users\gaeta\Desktop\HEI\3rd_year\Hackathon_301-302\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Best hyperparameters: {'learning_rate': 0.05, 'n_estimators': 1000}
Best score (roc_auc): 0.62
Train Accuracy: 0.63
Validation Accuracy: 0.63
Train F1-Score: 0.73
Validation F1-Score: 0.73
Train AUC-ROC: 0.63
Validation AUC-ROC: 0.62


In [35]:
y_proba_test = gb_clf.predict_proba(X_test)[:, 1]

auc, roc = get_auc_roc(y_test, y_proba_test)

export_model_data_json("gradientBoosting", auc, roc)